# Problema X20B1T0 — Solução via Funções de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x20b10t0.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Meio semi-infinito, com:

- **Contorno $x = 0$** (tipo 2): fluxo de calor prescrito $q''_0$
- **Condição em $x \to \infty$**: temperatura não perturbada
- **Condição inicial:** $T(x,0) = 0$

A notação X20 indica: geometria plana semi-infinita (X), fluxo prescrito em $x=0$ (2) e domínio semi-infinito em $x\to\infty$ (0).

## Solução analítica

Como $T(x,0) = 0$, a solução em termos de funções de Green é:

$$\Theta(x,t) = \frac{q''_0}{k}\,(4\alpha t)^{1/2}\,\operatorname{ierfc}\!\left(\frac{x}{(4\alpha t)^{1/2}}\right)$$

onde $\operatorname{ierfc}$ é a integral da função erro complementar:

$$\operatorname{ierfc}(u) = \frac{e^{-u^2}}{\sqrt{\pi}} - u\,\operatorname{erfc}(u)$$

Substituindo $u = x/(4\alpha t)^{1/2}$, a forma explícita é:

$$\Theta(x,t) = \frac{q''_0}{k}\,(4\alpha t)^{1/2} \left( \frac{e^{-x^2/(4\alpha t)}}{\sqrt{\pi}} - \frac{x}{(4\alpha t)^{1/2}}\,\operatorname{erfc}\!\left(\frac{x}{(4\alpha t)^{1/2}}\right) \right)$$

## Bibliotecas utilizadas

Para avaliar a solução analítica computacionalmente, utilizamos três bibliotecas Python:

- **NumPy** (`numpy`): biblioteca para computação numérica. Fornece vetores e matrizes eficientes e funções matemáticas como `np.sqrt`, `np.exp`, `np.linspace`.
- **Matplotlib** (`matplotlib.pyplot`): biblioteca para criação de gráficos 2D.
- **SciPy** (`scipy.special`): biblioteca científica. Usamos `erfc`, a função erro complementar, que não está disponível diretamente no NumPy.

A linha `import numpy as np` importa a biblioteca e cria o atalho `np` — assim escrevemos `np.sqrt` em vez de `numpy.sqrt`.

In [ ]:
import numpy as np                  # computação numérica
import matplotlib.pyplot as plt     # gráficos
from scipy.special import erfc      # função erro complementar: erfc(u)

## Parâmetros do problema

Os parâmetros físicos são armazenados como **variáveis** Python. Uma variável é criada com o sinal `=`, por exemplo `L = 66e-4` define $L = 0{,}0066$ m (a notação `66e-4` é equivalente a $66 \times 10^{-4}$). Os comentários após `#` não são executados e servem apenas para documentar o código.

In [ ]:
L     = 66e-4        # comprimento de referência [m]  (66e-4 = 0,0066 m)
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e3          # fluxo de calor imposto em x=0 [W/m²]  (1e3 = 1000)

## Vetores de tempo e posição

Para avaliar $T(x,t)$ precisamos de um conjunto de valores de tempo e de posição.

- `np.linspace(inicio, fim, N)` cria um vetor com `N` valores igualmente espaçados entre `inicio` e `fim`. Por exemplo, `np.linspace(0, 300, 5)` retorna `[0, 75, 150, 225, 300]`.
- Iniciamos em `1e-6` (um microssegundo) em vez de zero para evitar divisão por zero na fórmula.
- As posições `x_vals` são uma **lista** Python (delimitada por `[` e `]`). A lista `x_names` contém os rótulos correspondentes para a legenda do gráfico.

In [ ]:
t_max = 300                              # tempo máximo de simulação [s]
t     = np.linspace(1e-6, t_max, 600)   # 600 instantes de tempo entre 0 e t_max

x_vals  = [0, L/4, L/2, 3*L/4, L]                             # posições avaliadas
x_names = ['$x = 0$', '$x = L/4$', '$x = L/2$', '$x = 3L/4$', '$x = L$']  # rótulos

## Definição das funções

Em Python, uma **função** é definida com a palavra-chave `def`, seguida do nome, dos parâmetros entre parênteses e de dois pontos. O corpo da função deve ser **indentado** (recuado com espaços ou Tab) — a indentação indica ao Python quais linhas pertencem à função. A palavra `return` especifica o valor devolvido.

```python
def nome_da_funcao(parametro):
    resultado = ...     # estas linhas estão dentro da função (indentadas)
    return resultado    # devolve o valor calculado
```

Quando os parâmetros são vetores NumPy (como `u` abaixo), as operações `np.exp`, `np.sqrt`, etc. são aplicadas a **todos os elementos de uma só vez**, sem necessidade de laço.

In [ ]:
def ierfc(u):
    """Integral da função erro complementar: ierfc(u) = exp(-u²)/√π - u·erfc(u)."""
    return np.exp(-u**2) / np.sqrt(np.pi) - u * erfc(u)   # u**2 eleva ao quadrado

def temperatura(x, t_vec):
    """Retorna T(x, t) para um valor x e um vetor de tempos t_vec."""
    u = x / np.sqrt(4 * alpha * t_vec)                      # argumento de ierfc
    return (q0 / k) * np.sqrt(4 * alpha * t_vec) * ierfc(u)

## Gráfico do campo de temperatura

O laço `for` usa `zip(x_vals, x_names)` para percorrer as duas listas ao mesmo tempo.
`zip` funciona como um zíper: emparelha o primeiro elemento de cada lista, depois o segundo, e assim por diante.
Assim, a cada passo do laço, `x` assume um valor de posição e `nome` assume o rótulo correspondente:
no 1.º passo `x = 0` e `nome = '$x = 0$'`; no 2.º, `x = L/4` e `nome = '$x = L/4$'`; e assim até o 5.º.

A função `ax.plot` recebe os vetores $t$ e $T(x,t)$ e o parâmetro `label` (o texto que aparece na legenda).

In [ ]:
fig, ax = plt.subplots()   # cria a figura e os eixos

for x, nome in zip(x_vals, x_names):   # para cada posição e seu rótulo
    ax.plot(t, temperatura(x, t), label=nome)   # plota T(x,t) vs t

ax.set_xlabel('Tempo (s)')                      # rótulo do eixo horizontal
ax.set_ylabel('T(x, t)  (°C)')                  # rótulo do eixo vertical
ax.set_title('Problema X20B1T0 — meio semi-infinito com fluxo imposto')
ax.legend()                                     # exibe a legenda
ax.grid(True)                                   # grade de fundo
plt.tight_layout()
plt.show()